In [ ]:
from pathlib import Path
import pandas as pd
from langdetect import DetectorFactory, detect
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 0
DATA_FILE = Path("playstore_results_with_descriptions.xlsx")
RESULT_FILE = Path("filtered_apps.csv")
EXCLUDED_FILE = Path("excluded_apps.csv")
SUMMARY_FILE = Path("filter_summary.txt")

df = pd.read_excel(DATA_FILE, dtype=str)
df["Rating_raw"] = df["Rating"]
df["Downloads_raw"] = df["Downloads"]
summary = [f"Initial rows: {len(df)}"]

In [ ]:
df = df[df["Free"].eq("Yes")].copy()
summary.append(f"Free apps: {len(df)}")
df["Downloads"] = pd.to_numeric(
    df["Downloads_raw"].str.replace(",", "", regex=False), errors="coerce"
)
df = df[df["Downloads"] > 10000].copy()
summary.append(f"Free apps with more than 10,000 downloads: {len(df)}")
df["Rating"] = pd.to_numeric(df["Rating_raw"], errors="coerce")
df = df[df["Rating"].ge(4)].copy()
summary.append(f"Rating >= 4: {len(df)}")

def is_english_description(text):
    try:
        return detect(str(text)[:3000]) == "en"
    except (TypeError, ValueError, LangDetectException):
        return False

df["description_english"] = df["Description"].fillna("").apply(is_english_description)
df = df[df["description_english"]].copy()
df = df.drop_duplicates(subset="App Name").copy()
summary.append(f"Unique apps with English descriptions: {len(df)}")

In [ ]:
non_utility_pattern = (
    r"(?i)(fishing clash|fishing master|ace fishing|hooked inc|"
    r"fishing planet|fishing and life|fishing superstars|creatures of the deep|"
    r"purple ocean psychic|oceans board game|quiz usa|studyge|sea battle|"
    r"sea of conquest|seahorse evolution|ocean cleaner idle eco tycoon|"
    r"gwent|fly fishing simulator|let.?s fish|sea fishing.*game|"
    r"boat fishing game|ocean fishing king|fishing frenzy|psychic readings|"
    r"idle eco|warship online|pirate war|evolution.*sea|board game)"
)
description_non_utility_pattern = (
    r"(?i)(casino|slot machine|play this game|arcade game|warship game|"
    r"idle tycoon|psychic readings)"
)
by_name = df["App Name"].str.contains(non_utility_pattern, na=False)
by_description = df["Description"].fillna("").str.contains(
    description_non_utility_pattern, na=False
)
df["excluded_non_utility"] = by_name | by_description
excluded = df[df["excluded_non_utility"]].copy()
df = df[~df["excluded_non_utility"]].copy()
summary.append(f"Excluded games and non-utility apps: {len(excluded)}")
summary.append(f"Final professional/utility apps: {len(df)}")
summary.append(f"Minimum retained rating: {df['Rating'].min():.2f}")
assert not (df["Rating"] < 4).any()

excluded.to_csv(EXCLUDED_FILE, index=False)
df.to_csv(RESULT_FILE, index=False)
SUMMARY_FILE.write_text("\n".join(summary) + "\n", encoding="utf-8")
print("\n".join(summary))